In [4]:
import pye57
import numpy as np
from pathlib import Path


def e57_to_ply(e57_path: str) -> str:
    """
    Convert an E57 point cloud file to PLY format.
    Saves the output in the same directory with the same filename but .ply extension.

    Args:
        e57_path: Path to the input .e57 file

    Returns:
        Path to the saved .ply file
    """
    e57_path = Path(e57_path)
    ply_path = e57_path.with_suffix(".ply")

    e57 = pye57.E57(str(e57_path))

    # Read all scans and concatenate
    all_xyz = []
    all_rgb = []
    has_color = True

    for scan_index in range(e57.scan_count):
        data = e57.read_scan(scan_index, ignore_missing_fields=True)

        xs = data["cartesianX"]
        ys = data["cartesianY"]
        zs = data["cartesianZ"]
        all_xyz.append(np.column_stack((xs, ys, zs)))

        if has_color and all(k in data for k in ("colorRed", "colorGreen", "colorBlue")):
            r = data["colorRed"]
            g = data["colorGreen"]
            b = data["colorBlue"]
            # Normalize to 0-255 uint8 if floats in [0, 1]
            if r.dtype.kind == "f":
                r = (r * 255).clip(0, 255).astype(np.uint8)
                g = (g * 255).clip(0, 255).astype(np.uint8)
                b = (b * 255).clip(0, 255).astype(np.uint8)
            all_rgb.append(np.column_stack((r, g, b)))
        else:
            has_color = False

    points = np.concatenate(all_xyz, axis=0)
    colors = np.concatenate(all_rgb, axis=0) if has_color else None

    n = len(points)

    with open(ply_path, "wb") as f:
        # Header
        header = [
            "ply",
            "format binary_little_endian 1.0",
            f"element vertex {n}",
            "property float x",
            "property float y",
            "property float z",
        ]
        if has_color:
            header += [
                "property uchar red",
                "property uchar green",
                "property uchar blue",
            ]
        header.append("end_header")
        f.write(("\n".join(header) + "\n").encode("ascii"))

        # Binary data
        if has_color:
            xyz32 = points.astype(np.float32)
            rgb8 = colors.astype(np.uint8)
            # Interleave xyz + rgb per vertex
            vertex_data = np.zeros(n, dtype=[
                ("x", np.float32), ("y", np.float32), ("z", np.float32),
                ("red", np.uint8), ("green", np.uint8), ("blue", np.uint8),
            ])
            vertex_data["x"] = xyz32[:, 0]
            vertex_data["y"] = xyz32[:, 1]
            vertex_data["z"] = xyz32[:, 2]
            vertex_data["red"]   = rgb8[:, 0]
            vertex_data["green"] = rgb8[:, 1]
            vertex_data["blue"]  = rgb8[:, 2]
        else:
            vertex_data = np.zeros(n, dtype=[
                ("x", np.float32), ("y", np.float32), ("z", np.float32),
            ])
            xyz32 = points.astype(np.float32)
            vertex_data["x"] = xyz32[:, 0]
            vertex_data["y"] = xyz32[:, 1]
            vertex_data["z"] = xyz32[:, 2]

        f.write(vertex_data.tobytes())

    print(f"Saved {n:,} points → {ply_path}")
    return str(ply_path)

In [5]:
ply_path = e57_to_ply("/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/cup/cup_3.e57")

Saved 2,048 points → /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/cup/cup_3.ply


In [2]:
import os
import pye57
import numpy as np

def merge_e57_files(directory: str) -> str:
    """
    Merges all .e57 files in a directory into a single 'merged_pc.e57' file.
    
    Args:
        directory: Path to the directory containing .e57 files
        
    Returns:
        Path to the merged output file
    """
    e57_files = [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(".e57")
    ]

    if not e57_files:
        raise FileNotFoundError(f"No .e57 files found in {directory}")

    output_path = os.path.join(directory, "merged_pc.e57")

    # Collect all point data across all scans in all files
    all_x, all_y, all_z = [], [], []
    all_intensity, all_r, all_g, all_b = [], [], [], []
    has_intensity, has_color = True, True

    for file_path in e57_files:
        e57 = pye57.E57(file_path)
        for scan_index in range(e57.scan_count):
            data = e57.read_scan(scan_index, ignore_missing_fields=True)

            all_x.append(data["cartesianX"])
            all_y.append(data["cartesianY"])
            all_z.append(data["cartesianZ"])

            if has_intensity:
                if "intensity" in data:
                    all_intensity.append(data["intensity"])
                else:
                    has_intensity = False

            if has_color:
                if all(c in data for c in ("colorRed", "colorGreen", "colorBlue")):
                    all_r.append(data["colorRed"])
                    all_g.append(data["colorGreen"])
                    all_b.append(data["colorBlue"])
                else:
                    has_color = False

    # Concatenate everything
    merged = {
        "cartesianX": np.concatenate(all_x),
        "cartesianY": np.concatenate(all_y),
        "cartesianZ": np.concatenate(all_z),
    }
    if has_intensity and all_intensity:
        merged["intensity"] = np.concatenate(all_intensity)
    if has_color and all_r:
        merged["colorRed"] = np.concatenate(all_r)
        merged["colorGreen"] = np.concatenate(all_g)
        merged["colorBlue"] = np.concatenate(all_b)

    # Write merged file
    out_e57 = pye57.E57(output_path, mode="w")
    out_e57.write_scan_raw(merged)
    out_e57.close()

    print(f"Merged {len(e57_files)} file(s) into: {output_path}")
    return output_path

In [3]:
lol = merge_e57_files("/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/figures/gear_chatpgt/scans")

Merged 4 file(s) into: /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/figures/gear_chatpgt/scans/merged_pc.e57


In [2]:
import os
import numpy as np
import trimesh
import pye57


def sample_obj_to_e57(obj_path: str, num_points: int = 2048) -> str:
    """
    Samples a uniform point cloud from an OBJ file and saves it as an E57 file.

    Args:
        obj_path:   Path to the input .obj file
        num_points: Number of points to sample (default: 2048)

    Returns:
        Path to the output .e57 file
    """
    mesh = trimesh.load(obj_path, force="mesh")

    points, _ = trimesh.sample.sample_surface_even(mesh, num_points)

    # Fallback in case even sampling returns fewer points than requested
    if len(points) < num_points:
        extra, _ = trimesh.sample.sample_surface(mesh, num_points - len(points))
        points = np.concatenate([points, extra], axis=0)

    output_path = os.path.splitext(obj_path)[0] + ".e57"

    data = {
        "cartesianX": points[:, 0].astype(np.float64),
        "cartesianY": points[:, 1].astype(np.float64),
        "cartesianZ": points[:, 2].astype(np.float64),
    }

    e57 = pye57.E57(output_path, mode="w")
    e57.write_scan_raw(data)
    e57.close()

    print(f"Saved {len(points)} points to: {output_path}")
    return output_path

In [3]:
sample_obj_to_e57("/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/cup/cup_3.obj")

Saved 2048 points to: /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/cup/cup_3.e57


'/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Karl-Kraus-Nachwuchsförderpreis/presentation/cup/cup_3.e57'